In [ ]:
!pip install --upgrade google-cloud-aiplatform[evaluation]

In [1]:
#init required libraries
import pandas as pd
import json
import vertexai
from io import StringIO
from vertexai.generative_models import Part, Tool, grounding
from vertexai.evaluation import EvalTask, PointwiseMetric, PairwiseMetric
import vertexai.evaluation.metrics.pointwise_metric as pointwise_metric

pd.options.display.max_colwidth = 300

vertexai.init(location="europe-west4")
def analyze_gemini(contents, model_name, instruction, response_mime, response_schema, token_limit, bUse_Grounding):
    from vertexai.generative_models import GenerationConfig, GenerativeModel, HarmCategory, HarmBlockThreshold
    def get_model():
        return GenerativeModel(model_name)
        #return GenerativeModel(model_name, system_instruction=instruction)
    generation_config=GenerationConfig(
        candidate_count = 1,
        max_output_tokens = token_limit,
        temperature = 0,
        top_p = 0.5,
        top_k = 1,
        response_mime_type = response_mime,
        response_schema = response_schema
    )

    tool_google_search = Tool.from_google_search_retrieval(grounding.GoogleSearchRetrieval())

    responses = get_model().generate_content(
        contents=contents,
        generation_config=generation_config,
        safety_settings={
            HarmCategory.HARM_CATEGORY_UNSPECIFIED: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        },
        stream=False, 
        tools=[tool_google_search] if bUse_Grounding else None
    )

    return responses.text

* LLM 을 이용해 번역 엔진을 만들고자 합니다. Human prefrence 가 있는 Reference 가 존재하는 경우 번역 결과에 대해 번역 품질을 LLM 이 평가할 수 있습니다.
* 이 평가 결과를 통해 LLM 엔진의 Prompt 를 개선하는 작업을 수행하면 됩니다.
* 개선된 Prompt 로 새로운 컨텐츠를 번역하고, 이 번역결과는 reference 가 없기 때문에 reference free 모드로 번역 품질을 살펴봅니다.

In [2]:
# Pointwise 검사를 위한 데이터셋 준비
# Pointwise는 단일 모델의 결과에 대한 평가입니다.
data = [
    ["This account is not protected with a two-factor authentication.", 
     "2차 인증이 되지 않은 계정입니다.",
     "이중 인증으로 보호되지 않은 계정입니다."],
    ["Even the longest journey begins with a single step.",
    "천 리 길도 한 걸음부터.",
    "가장 먼 여정도 한 걸음에서 시작됩니다."],
    ["Even the longest journey begins with a single step.",
    "Лиха беда начало.",
    "Даже самая длинная дорога продолжается с одного шага."]
]
pointwise_df = pd.DataFrame(data, columns=['prompt', 'reference', 'response'])
pointwise_df

,prompt,reference,response
0,This account is not protected with a two-factor authentication.,2차 인증이 되지 않은 계정입니다.,이중 인증으로 보호되지 않은 계정입니다.
1,Even the longest journey begins with a single step.,천 리 길도 한 걸음부터.,가장 먼 여정도 한 걸음에서 시작됩니다.
2,Even the longest journey begins with a single step.,Лиха беда начало.,Даже самая длинная дорога продолжается с одного шага.


In [3]:
from vertexai.evaluation import EvalTask, PointwiseMetric, PairwiseMetric
pointwise_ground_truth_metric_prompt = """
# Instruction
당신은 전문적인 번역 품질 평가자 입니다. 당신은 AI 모델이 번역한 내용의 품질을 평가해야 합니다.
우리는 당신에게 'Source, AI translation, Ground Truth'를 제공할 것입니다.
업무를 수행하기 위해 'Source, AI translation, Ground Truth'을 신중하게 읽고, 아래의 'Evaluation' 섹션에 정의된 'Evaluation criteria'에 근거하여 평가해야 합니다.
당신은 'Rating rubric'과 'Evaluation steps'에 기반하여 평가해야 합니다. 평가의 이유에 대해 단계별로 '한글'로 설명하고 'Rating rubric'의 순위만 선택해야 합니다.

# Evaluation
## Metric Definition
번역이 'Ground Truth'와 동일하게 번역되었는지, 그렇지 않으면 의미적으로 유사하고 간결, 명확, 캐주얼하게 작성되었는지 전반적으로 확인합니다.

## Evaluation criteria
예시 일치성: 예시로 제공되는 'Ground Truth'와 얼마나 일치하는지 확인합니다.
지시를 잘 따르는지: 'AI translation'이 'Evaluation Definition'을 잘 따라서 수행됐는지 확인합니다.

## Rating rubric
5: (매우 좋음). 'Ground Truth'와 글자수 등 표현이 정확하게 일치함
4: (좋음). 'Ground Truth'와 어순 외 차이 없음
3: (괜찮음). 속담이나 관용적 표현을 사용하지 못했지만 이해하는데 큰 문제 없음
2: (나쁨). 이해하기가 어려움
1: (매우 나쁨). 번역결과가 'Ground Truth'와 매우 다름

## Evaluation steps
STEP 1: 'Ground Truth'을 참고하여 'AI translation'이 번역한 내용을 비교합니다.
STEP 2: 평가 규칙에 따라 점수를 부여합니다.

# Source, AI translation, Ground Truth
### Source
{prompt}

### Ground Truth
{reference}

### AI translation
{response}
"""

pointwise_ground_truth_text_quality = PointwiseMetric(
    metric="pointwise_ground_truth_quality",
    metric_prompt_template=pointwise_ground_truth_metric_prompt,
)
#For MetricX, require source field
pointwise_df['source'] = pointwise_df['prompt']
eval_task = EvalTask(dataset=pointwise_df, 
                    metrics=[pointwise_ground_truth_text_quality, "bleu", "rouge", pointwise_metric.MetricX(), pointwise_metric.Comet()],
                    experiment="pointwise-eval")
result_pointwise = eval_task.evaluate()

#comet, higher is better (0-1)
#metricx, lower is better (0-25)
result_pointwise.metrics_table

Associating projects/1045259343465/locations/europe-west4/metadataStores/default/contexts/pointwise-eval-67ff81ea-92db-4e85-9b1b-2ac1fa2c14a0 to Experiment: pointwise-eval


Computing metrics with a total of 15 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 15/15 [00:17<00:00,  1.19s/it]

All 15 metric requests are successfully computed.
Evaluation Took:17.82862689999456 seconds


,prompt,reference,response,source,pointwise_ground_truth_quality/explanation,pointwise_ground_truth_quality/score,bleu/score,rouge/score,metricx/score,comet/score
0,This account is not protected with a two-factor authentication.,2차 인증이 되지 않은 계정입니다.,이중 인증으로 보호되지 않은 계정입니다.,This account is not protected with a two-factor authentication.,"STEP 1: 'Ground Truth'와 'AI translation'을 비교했습니다.\n'Ground Truth'는 ""2차 인증이 되지 않은 계정입니다.""이고, 'AI translation'은 ""이중 인증으로 보호되지 않은 계정입니다.""입니다.\n\nSTEP 2: 평가 규칙에 따라 점수를 부여합니다.\n'AI translation'이 'Ground Truth'와 의미적으로 유사하고 간결, 명확하게 작성되었습니다. 하지만 'Ground Truth'에서 '2차 인증'이라고 명시했는데, 'AI translation'은 '이중 ...",4.0,0.302138,0.0,0.961893,0.915181
1,Even the longest journey begins with a single step.,천 리 길도 한 걸음부터.,가장 먼 여정도 한 걸음에서 시작됩니다.,Even the longest journey begins with a single step.,"STEP 1: 'AI translation'에서 ""가장 먼 여정도 한 걸음에서 시작됩니다.""라고 번역했고, 'Ground Truth'에서는 ""천 리 길도 한 걸음부터.""라고 번역했습니다.\nSTEP 2: 'AI translation'은 'Ground Truth'와 비교했을 때 속담을 활용하지 못했고, 의미가 다르게 번역되었습니다. 따라서 '나쁨'에 해당하는 2점을 부여합니다.",2.0,0.078099,0.0,0.964249,0.925086
2,Even the longest journey begins with a single step.,Лиха беда начало.,Даже самая длинная дорога продолжается с одного шага.,Even the longest journey begins with a single step.,"STEP 1: AI 모델은 ""Even the longest journey begins with a single step.""을 ""Даже самая длинная дорога продолжается с одного шага.""로 번역했습니다. Ground Truth는 ""Лиха беда начало.""입니다.\nSTEP 2: AI 번역은 Ground Truth와 완전히 다릅니다. AI는 직역을 했지만, Ground Truth는 ""시작이 반이다""라는 의미의 속담으로 번역했습니다. 따라서 ""번역결과가 Ground Truth와 매우...",1.0,0.047677,0.0,5.247432,0.568027


In [4]:
# Pairwise 검사를 위한 데이터셋 준비
# Pairwise 는 2개 모델의 성능을 비교하기 위한 방법입니다.
#prompt(Source), reference, response (Response B), baseline_model_response (Response A)
data = [
    ["This account is not protected with a two-factor authentication.", 
     "2차 인증이 되지 않은 계정입니다.",
     "이중 인증으로 보호되지 않은 계정입니다.",
     "이 계정은 이중 인증으로 안전하게 보호되지 않습니다."],
    ["Even the longest journey begins with a single step.",
    "천 리 길도 한 걸음부터.",
    "가장 먼 여정도 한 걸음에서 시작됩니다.",
    "가장 긴 여행조차도 한 걸음부터 시작됩니다."],
    ["Even the longest journey begins with a single step.",
    "Лиха беда начало.",
    "Даже самая длинная дорога продолжается с одного шага.",
    "Даже самый долгий путь начинается с первого шага."]
]
pairwise_df = pd.DataFrame(data, columns=['prompt', 'reference', 'response', 'baseline_model_response'])
pairwise_df

,prompt,reference,response,baseline_model_response
0,This account is not protected with a two-factor authentication.,2차 인증이 되지 않은 계정입니다.,이중 인증으로 보호되지 않은 계정입니다.,이 계정은 이중 인증으로 안전하게 보호되지 않습니다.
1,Even the longest journey begins with a single step.,천 리 길도 한 걸음부터.,가장 먼 여정도 한 걸음에서 시작됩니다.,가장 긴 여행조차도 한 걸음부터 시작됩니다.
2,Even the longest journey begins with a single step.,Лиха беда начало.,Даже самая длинная дорога продолжается с одного шага.,Даже самый долгий путь начинается с первого шага.


In [5]:
pairwise_model_compare_metric_prompt = """
# Instruction
당신은 전문적인 번역 품질 평가자 입니다. 당신은 2개의 AI 모델이 번역한 내용의 품질을 평가해야 합니다.
우리는 당신에게 'Source, Response A, Response B, Ground Truth'를 제공할 것입니다.
업무를 수행하기 위해 'Source, Response A, Response B, Ground Truth'을 신중하게 읽고, 아래의 '평가' 섹션에 정의된 '평가항목'에 근거하여 평가해야 합니다.
당신은 'Evaluation rule'과 'Evaluation steps'에 기반하여 평가해야 합니다. 평가의 이유에 대해 단계별로 '한글'로 설명하고 'Evaluation rule'의 순위만 선택해야 합니다.

# Evaluation
## Metric Definition
번역이 'Ground Truth'와 동일하게 번역되었는지, 그렇지 않으면 의미적으로 유사하고 간결, 명확, 캐주얼하게 작성되었는지 전반적으로 확인합니다.

## Evaluation criteria
예시 일치성: 예시로 제공되는 'Ground Truth'와 얼마나 일치하는지 확인합니다.
지시를 잘 따르는지: 'AI translation'이 'Evaluation Definition'을 잘 따라서 수행됐는지 확인합니다.

## Rating rubric
STEP 1: Analyze Response A based on all the Criteria.
STEP 2: Analyze Response B based on all the Criteria.
STEP 3: Compare the overall performance of Response A and Response B based on your analyses and assessment.
STEP 4: Output your preference of "A", "SAME" or "B" to the pairwise_choice field according to the Rating Rubric.
STEP 5: Output your assessment reasoning in the explanation field.

# Source, Response A, Response B, Ground Truth
### Source
{prompt}

### Ground Truth
{reference}

### Response A
{baseline_model_response}

### Response B
{response}
"""

pairwise_ground_truth_text_quality = PairwiseMetric(
    metric="pairwise_ground_truth_quality",
    metric_prompt_template=pairwise_model_compare_metric_prompt,
)

eval_task = EvalTask(dataset=pairwise_df, 
                    metrics=[pairwise_ground_truth_text_quality],
                    experiment="pairwise-eval")
result_pairwise = eval_task.evaluate()
result_pairwise.metrics_table.rename(columns={"baseline_model_response": "Response A", "response": "Response B"}).replace(['BASELINE', 'CANDIDATE'], ['A', 'B'])

Associating projects/1045259343465/locations/europe-west4/metadataStores/default/contexts/pairwise-eval-5be01ee3-ade0-442f-811d-0613b0297a6a to Experiment: pairwise-eval


Computing metrics with a total of 3 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 3/3 [00:06<00:00,  2.30s/it]

All 3 metric requests are successfully computed.
Evaluation Took:6.92804199999955 seconds


,prompt,reference,Response B,Response A,pairwise_ground_truth_quality/explanation,pairwise_ground_truth_quality/pairwise_choice
0,This account is not protected with a two-factor authentication.,2차 인증이 되지 않은 계정입니다.,이중 인증으로 보호되지 않은 계정입니다.,이 계정은 이중 인증으로 안전하게 보호되지 않습니다.,"Response A는 '이중 인증으로 안전하게 보호되지 않습니다.'라고 번역했는데, Ground Truth와 비교했을 때 '안전하게'라는 불필요한 단어가 추가되었습니다. Response B는 '이중 인증으로 보호되지 않은 계정입니다.'라고 번역했는데, Ground Truth와 의미적으로 동일하며 더 간결하고 자연스럽습니다. 따라서 Response B가 Ground Truth에 더 가깝다고 판단했습니다.",B
1,Even the longest journey begins with a single step.,천 리 길도 한 걸음부터.,가장 먼 여정도 한 걸음에서 시작됩니다.,가장 긴 여행조차도 한 걸음부터 시작됩니다.,"Response A와 B 모두 자연스러운 번역이지만, Response B가 Source의 'longest journey'의 의미를 '가장 먼 여정'으로 의역하여 Ground Truth의 '천 리 길'에 더 가깝게 표현했기 때문입니다. Response A는 '가장 긴 여정'으로 직역하여 Ground Truth와 의미적 거리가 있습니다.",B
2,Even the longest journey begins with a single step.,Лиха беда начало.,Даже самая длинная дорога продолжается с одного шага.,Даже самый долгий путь начинается с первого шага.,"Response A는 '가장 긴 여정도 한 걸음부터 시작된다'라는 속담의 의미를 잘 전달하고 있으며, Ground Truth가 의도하는 의미와 유사한 러시아 속담을 제시하고 있습니다. Response B는 '가장 긴 길도 한 걸음부터 계속된다'로 해석되어 의미가 어색합니다. 따라서 Ground Truth의 의도와 유사성, 자연스러움 측면에서 Response A가 더 적절합니다.",A


* Reference 가 없는, 새로운 컨텐츠에 대한 번역과 이를 MetricX를 이용해 reference free (Quality Estimation, QE) 모드로 평가하는 예시입니다.

In [6]:
prompt = """당신을 비디오를 분석해서 transcript를 작성해야 하는 AI Assistant 입니다.
아래 가이드라인에 맞게 transcription을 작성해주세요.

1. 첨부된 비디오를 분석하여 아래와 같은 포맷으로 모든 대화 내용을 빠짐없이 출력해주세요.
2. 결과 출력 단위는 비디오 내의 장면이 구분되는 특정 장소를 기준으로 나누어서 출력해주세요.
3. 목소리를 기반으로 화자(speaker)를 정확하게 분리해서 영어로 출력해주세요.
4. 목소리외에 다양한 효과음, 감정표현은 괄호를 사용해서 반드시 자세히 표현해주세요."""
response_schema = {
    "type": "ARRAY",
    "items": {
        "type": "OBJECT",
        "properties": {
            "location": { "type": "STRING",},
            "start_time": { "type": "STRING",},
            "end_time": { "type": "STRING",},
            "elapsed_time": { "type": "STRING",},
            "transcription": {
                "type": "ARRAY",
                "items" : {
                  "type": "OBJECT",
                  "properties": {
                    "speaker": { "type": "STRING",},
                    "transcript": { "type": "STRING",},
                  }
                }
            },
        },
        "required": ["start_time","end_time","elapsed_time"],
    },
}
video = Part.from_uri(mime_type="video/*", uri="https://www.youtube.com/watch?v=OoUVSHDbAeM")
content = [
    prompt,
    video
]
response = analyze_gemini(content, "gemini-1.5-pro-002", "", "application/json", response_schema, 8192, False)
pd_transcription = pd.json_normalize(json.loads(response), record_path='transcription', meta=['location', 'start_time', 'end_time', 'elapsed_time'])
pd_transcription

,speaker,transcript,location,start_time,end_time,elapsed_time
0,Isobel,"Come on, Stephen.",Train Station,00:00:05,00:00:26,00:00:21
1,Isobel,Got to move on.,Train Station,00:00:05,00:00:26,00:00:21
2,Man,"It's moving, you, ma'am.",Train Station,00:00:05,00:00:26,00:00:21
3,Man,"Chop, chop.",Train Station,00:00:05,00:00:26,00:00:21
4,Train Staff,(Whistle),Train Station,00:00:05,00:00:26,00:00:21
5,Train,(Train Horn),Train Station,00:00:05,00:00:26,00:00:21
6,Professor,"A star,",Classroom,00:00:27,00:01:27,00:01:00
7,Professor,"more than three times the size of our sun, ought to end its life how?",Classroom,00:00:27,00:01:27,00:01:00
8,Professor,With a collapse.,Classroom,00:00:27,00:01:27,00:01:00
9,Professor,"The gravitational forces of the entire mass overcoming the electromagnetic forces of individual atoms, and so collapsing inwards.",Classroom,00:00:27,00:01:27,00:01:00


In [9]:
source_text = pd_transcription[['location', 'speaker', 'transcript']].to_csv(index=False)
content = [
    video,
    """
    다음은 위 영상에 대해 CSV 로 만들어진 영화 대사입니다. CSV 헤더에는 'location', 'speaker', 'transcript' 로 구성돼 있습니다.
    'transcript'의 내용을 영상의 내용을 참고하여 번역하여 'translated' 에 결과를 알려주세요.
    예시의 CSV 포맷으로 헤더를 포함하여 출력해 주세요.
    example:
    location,speaker,transcript,translated
    Restraunt,Mike,"Hello, Good Day!","안녕하세요 좋은 하루입니다!"
    """,
    source_text
]
output = analyze_gemini(content, "gemini-1.5-pro-002", "", "text/plain", None, 8192, False)
result = pd.read_csv(StringIO(output))
result

,location,speaker,transcript,translated
0,Train Station,Isobel,"Come on, Stephen.","스티븐, 어서."
1,Train Station,Isobel,Got to move on.,어서 가야 해.
2,Train Station,Man,"It's moving, you, ma'am.","움직이고 있습니다, 부인."
3,Train Station,Man,"Chop, chop.",서둘러.
4,Train Station,Train Staff,(Whistle),(휘파람)
5,Train Station,Train,(Train Horn),기차 경적
6,Classroom,Professor,"A star,",별은
7,Classroom,Professor,"more than three times the size of our sun, ought to end its life how?",태양보다 세 배 이상 큰 별은 어떻게 수명을 다할까요?
8,Classroom,Professor,With a collapse.,붕괴로.
9,Classroom,Professor,"The gravitational forces of the entire mass overcoming the electromagnetic forces of individual atoms, and so collapsing inwards.",전체 질량의 중력이 개별 원자의 전자기력을 극복하고 안쪽으로 붕괴됩니다.


In [10]:
eval_task = EvalTask(dataset=result[['transcript', 'translated']].rename(columns={'transcript': 'source', 'translated': 'response'}),
                    metrics=[pointwise_metric.MetricX(version="METRICX_24_SRC")],
                    experiment="pointwise-eval")
result_pointwise = eval_task.evaluate()
result_pointwise.metrics_table

#metricx, lower is better (0-25)
#From the paper, https://aclanthology.org/2024.wmt-1.35/
# 0 ~ 1 is good
# above 5 means undertranslation

Associating projects/1045259343465/locations/europe-west4/metadataStores/default/contexts/pointwise-eval-54a72d56-2e03-4272-be2b-d1d571e75eaf to Experiment: pointwise-eval


Computing metrics with a total of 54 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 54/54 [00:58<00:00,  1.08s/it]

All 54 metric requests are successfully computed.
Evaluation Took:58.25457330000063 seconds


,source,response,metricx/score
0,"Come on, Stephen.","스티븐, 어서.",3.070616
1,Got to move on.,어서 가야 해.,2.760401
2,"It's moving, you, ma'am.","움직이고 있습니다, 부인.",5.795606
3,"Chop, chop.",서둘러.,7.127749
4,(Whistle),(휘파람),0.731615
5,(Train Horn),기차 경적,2.611493
6,"A star,",별은,2.492091
7,"more than three times the size of our sun, ought to end its life how?",태양보다 세 배 이상 큰 별은 어떻게 수명을 다할까요?,1.987360
8,With a collapse.,붕괴로.,5.083920
9,"The gravitational forces of the entire mass overcoming the electromagnetic forces of individual atoms, and so collapsing inwards.",전체 질량의 중력이 개별 원자의 전자기력을 극복하고 안쪽으로 붕괴됩니다.,3.057247
